In [1]:
import os
import sys

CURRENT_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, os.pardir))
LLM_DIR = os.path.join(PROJECT_ROOT, "llm_results")  # ou "llm_results/arara" se for o seu caso
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:

database = 'neo4j'
query_type = 'condition'
dataset_type = 'ImplicitQuery'  # ou 'ExplicitQuery' dependendo do seu caso
groundtruths = os.path.join(PROJECT_ROOT, "dataset", "movie", f"{dataset_type}.json")

import os, json, itertools

def is_candidate(file_name: str, eval_type: str) -> bool:
    return (
        eval_type in file_name
        and ("output" in file_name or "prediction" in file_name)
        and file_name.endswith(".jsonl")
    )

def validate_predictions_file(path: str):
    if not os.path.isfile(path):
        return False, "arquivo não existe"
    if os.path.getsize(path) == 0:
        return False, "arquivo vazio (0 bytes)"
    # lê a primeira linha não-vazia
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                return False, f"JSON inválido na 1ª linha não-vazia: {e}"
            if not isinstance(obj, dict):
                return False, "1ª linha não é um objeto JSON"
            if "id" not in obj or "response" not in obj:
                return False, "faltam chaves obrigatórias: 'id' e/ou 'response'"
            return True, "ok"
    return False, "somente linhas vazias"

# --- use ---
# garanta que dataset_type exista (ex.: "ImplicitQuery" ou "ExplicitQuery")
# dataset_type = "ImplicitQuery"
eval_type = f"movie-{dataset_type}"

avaliados = 0
pulados = 0

for root, dirs, files in os.walk(LLM_DIR):
    print("root=", root)
    for file in files:
        print("file=", file)
        if not is_candidate(file, eval_type):
            continue

        predictions = os.path.join(root, file)
        ok, msg = validate_predictions_file(predictions)
        if not ok:
            print(f"🟡 Pulando: {predictions} — {msg}")
            pulados += 1
            continue

        print(f"✅ Avaliando: {predictions}")
        scrit_name = (
            f'eval_movie.py --database {database} '
            f'--query_type {query_type} '
            f'--groundtruths "{groundtruths}" '
            f'--predictions "{predictions}"'
        )
        get_ipython().run_line_magic('run', scrit_name)
        print('---------------------------------------------')
        avaliados += 1

print(f"\nResumo: {avaliados} avaliados, {pulados} pulados.")



root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results
file= .DS_Store
file= grpo-predictions.jsonl
file= grpo-predictions_output.jsonl
root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/gpt-4o-k
file= movie-ExplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-MisinformedQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ItemBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ExplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/gpt-4o-k/movie-ImplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatc

INFO:root:Successfully connected to the Neo4j database.
Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6338722937791175
INFO:root: Ftr: 0.0027404768429706767
INFO:root: Recall: 0.24486544179610772
INFO:root: Precision: 0.12277336256508632
INFO:root: Ndcg: 0.20903464947029457
INFO:root: Satisfied Ratio: 0.2657332590322281
INFO:root: Existence In Kg Ratio: 0.7950123321457934
INFO:root:Successfully connected to the Neo4j database.


---------------------------------------------
file= movie-MisinformedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ItemBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-UserBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/gpt-4o-k/movie-ImplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl


Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6338722937791175
INFO:root: Ftr: 0.0002740476842970677
INFO:root: Recall: 0.253276303337964
INFO:root: Precision: 0.12726774458755824
INFO:root: Ndcg: 0.21435153383375252
INFO:root: Satisfied Ratio: 0.2678092330465353
INFO:root: Existence In Kg Ratio: 0.8138668128254317
INFO:root:Successfully connected to the Neo4j database.


---------------------------------------------
file= movie-MisinformedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-MisinformedQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-UserBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/llama-3.1-70b-instruct
file= book-ExplicitQuery_llama-3.1-70b-instruct-prediction.jsonl
file= movie-UserBasedQuery_llama-3.1-70b-instruct-prediction.jsonl
file= movie-ItemBasedQuery_llama-3.1-70b-instruct-prediction.jsonl
file= movie-ExplicitQuery_llama-3.1-70b-instruct-prediction.jsonl
file= movie-ImplicitQuery_llama-3.1-70b-instruct-prediction.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/llama-3.1-70b-instruct/movie-ImplicitQuery_llama-3.1-70b-instruct-prediction.jsonl


...

---------------------------------------------
file= movie-ExplicitQuery_arara_gpt_41_historyTrue-prediction.jsonl
root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022-k
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ItemBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022-k/movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl


Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6338722937791175
INFO:root: Ftr: 0.000822143052891203
INFO:root: Recall: 0.26988350402984546
INFO:root: Precision: 0.13669498492737736
INFO:root: Ndcg: 0.23652698776054584
INFO:root: Satisfied Ratio: 0.25765170292848616
INFO:root: Existence In Kg Ratio: 0.8684023020005481
INFO:root:Successfully connected to the Neo4j database.


---------------------------------------------
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-UserBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/r

Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6338722937791175
INFO:root: Ftr: 0.009865716634694436
INFO:root: Recall: 0.24698256890638365
INFO:root: Precision: 0.12600712523979174
INFO:root: Ndcg: 0.21948592927206143
INFO:root: Satisfied Ratio: 0.27668895996898324
INFO:root: Existence In Kg Ratio: 0.7867909016168814


---------------------------------------------
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ItemBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-UserBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl

Resumo: 18 avaliados, 1 pulados.
